# EDA Part 2: Raw Image Analysis

**Author:** Shreya Aggarwal  
**Date:** 2026-02-22  
**GitHub:** https://github.com/shreya1297/Thesis

---

## What is this notebook?

Now that we know *how many* samples we have (Part 1), let's look at the actual images.

This notebook answers:
- What do the microscopy images actually look like?
- How bright/dim are the different channels — and is this consistent across samples?
- Can we *visually* tell the difference between Quiescent and TNF-stimulated cells?
- Are there any images with quality problems (too dark, too noisy)?

## Connection to Research Questions

Good image quality is a prerequisite for everything else. If a channel is too dim,
segmentation fails. If brightness varies wildly between samples, comparisons are unfair.

This notebook is the sanity check: **can we trust the raw data?**

---

## The 3 Channels — a quick reminder

Each microscopy image has **3 channels** (think: 3 different colour photographs of the same dish).
Each channel lights up a different protein:

| Channel | Protein stained | What it shows | Why we care |
|---------|----------------|---------------|-------------|
| **Ch1** | VE-cadherin (GFP, green) | Cell-cell junctions — the "glue" between cells | Tells us about adhesion energy (CPM parameter J) |
| **Ch2** | RhoA or RhoB (mCherry, red) | Rho GTPase activity in the cell body | Main biological variable we want to quantify (SQ3) |
| **Ch3** | SiR-Actin (far-red) | Entire cell body (actin skeleton outlines the whole cell) | **Best channel for segmentation** — most complete boundary |


In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 10,
})

# ── Find project root ─────────────────────────────────────────────────────
_p = Path().resolve()
PROJECT_ROOT = None
for _parent in [_p] + list(_p.parents):
    if (_parent / 'Quantification_EndothelialCells_CPM').is_dir():
        PROJECT_ROOT = _parent
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError('Cannot find project root. Run from inside implementation/ folder.')

THESIS_DIR = PROJECT_ROOT / 'Quantification_EndothelialCells_CPM'
DATA_DIR   = PROJECT_ROOT / 'data' / 'JK2513_A1R_HUVEC_dataset cellular pots model'
OUTPUT_DIR = THESIS_DIR / 'output'
EDA_DIR    = OUTPUT_DIR / 'eda' / 'eda_02'
EDA_DIR.mkdir(parents=True, exist_ok=True)

print(f'Data : {DATA_DIR}')
print(f'EDA  : {EDA_DIR}')

# ── Helper: load 16-bit TIFF ──────────────────────────────────────────────
def load_tiff(path) -> np.ndarray:
    """Load a 16-bit TIFF and return as numpy array (dtype=uint16)."""
    return np.array(Image.open(path))

def display_normalise(img: np.ndarray, pct: float = 98.0) -> np.ndarray:
    """
    Stretch image contrast for display.
    Clips at the given percentile (e.g. 98%) to avoid outlier bright spots
    dominating the display, then rescales to [0, 1].
    This does NOT change the underlying data — only for visualization.
    """
    lo, hi = img.min(), np.percentile(img, pct)
    if hi == lo:
        return np.zeros_like(img, dtype=float)
    return np.clip((img.astype(float) - lo) / (hi - lo), 0, 1)

# ── Helper: discover all samples ──────────────────────────────────────────
def discover_samples(data_dir: Path) -> list:
    """Scan data folder and return list of sample dicts with T0001 paths."""
    samples = []
    for ch3_dir in sorted(data_dir.rglob('*_Channel3')):
        if 'Out of focus' in str(ch3_dir):
            continue
        sample_name = ch3_dir.name.replace('_Channel3', '')
        folder_name = ch3_dir.parent.name
        frames = sorted(ch3_dir.glob('*.tif'))
        if not frames:
            continue
        t0001 = [p for p in frames if 'T0001' in p.name]
        if not t0001:
            continue
        protein   = 'RhoA' if ('mChA' in sample_name or 'mCHA' in sample_name) else 'RhoB'
        condition = 'TNF' if 'TNF' in sample_name else 'Quiescent'
        ch1_dir = ch3_dir.parent / ch3_dir.name.replace('Channel3', 'Channel1')
        ch2_dir = ch3_dir.parent / ch3_dir.name.replace('Channel3', 'Channel2')
        ch1 = sorted(ch1_dir.glob('*T0001*.tif')) if ch1_dir.exists() else []
        ch2 = sorted(ch2_dir.glob('*T0001*.tif')) if ch2_dir.exists() else []
        samples.append({
            'folder': folder_name, 'sample_name': sample_name,
            'protein': protein, 'condition': condition,
            'n_frames': len(frames),
            'ch1': ch1[0] if ch1 else None,
            'ch2': ch2[0] if ch2 else None,
            'ch3': t0001[0],
        })
    return samples

# Discover all samples
samples = discover_samples(DATA_DIR)
print(f'Found {len(samples)} samples')

## 1. One Example: All 3 Channels Side by Side

Let's start by looking at what one sample looks like in each of the 3 channels.

We use sample `26112025_Q_003` (RhoA, Quiescent) as our first example.

> **Why do the images look different from what you might expect?**  
> The microscope camera is very sensitive — it captures very faint differences in
> brightness. The images look dim because the screen assumes values go up to 65,535
> but our sensor only uses values up to ~4,095. We apply a contrast stretch for display.

In [ ]:
# Pick one example sample (RhoA, Quiescent, from the first experiment date)
example = next(
    (s for s in samples if 'Q_003' in s['sample_name'] and 'RhoA' in s['protein']
     and '26112025' in s['sample_name']), samples[0]
)
print(f"Using sample: {example['sample_name']}")

ch_paths = [
    ('Ch1 — VE-cadherin\n(cell-cell junctions)', example['ch1']),
    ('Ch2 — RhoA GTPase\n(cytoplasmic activity)', example['ch2']),
    ('Ch3 — SiR-Actin\n(cell body / skeleton)', example['ch3']),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle(f"Example: {example['sample_name']} | T0001",
             fontsize=11, fontweight='bold')

for ax, (title, path) in zip(axes, ch_paths):
    if path is None:
        ax.text(0.5, 0.5, 'File not found', ha='center', va='center',
                transform=ax.transAxes)
        ax.set_title(title)
        continue
    img = load_tiff(path)
    disp = display_normalise(img, pct=99)
    ax.imshow(disp, cmap='gray')
    ax.set_title(title, fontsize=9)
    ax.axis('off')
    # Add pixel value range annotation
    ax.text(5, img.shape[0] - 20,
            f'Raw range: {img.min()}–{img.max()}',
            color='white', fontsize=8, va='bottom')

plt.tight_layout()
plt.savefig(EDA_DIR / 'raw_example_3channels.png', bbox_inches='tight', dpi=150)
plt.show()
print(f'Saved → {EDA_DIR / "raw_example_3channels.png"}')

## 2. The 16-bit TIFF: Why Images Look Dark

Understanding the raw pixel values is important before we do any analysis.

**The sensor:**  
The microscope uses a 12-bit camera → pixel values range from **0 to 4,095**.
FIJI exports these as 16-bit TIFF files (which can hold values 0–65,535).
The top ~61,000 values are never used — they're just empty.

**Why this matters for segmentation:**  
16-bit TIFFs preserve all the fine intensity differences in the 0–4,095 range.
If we had converted to 8-bit PNG (0–255) first, we'd lose detail.
This is why switching from PNG → TIFF improved segmentation coverage from **87% → 96%**.

**For display (not analysis):**  
We normalise to the 98th percentile so the image fills the full brightness range
and is visible. This is display-only and does not affect segmentation.


In [ ]:
# Show intensity histogram for one channel, raw vs display-normalised
img = load_tiff(example['ch3'])
p98 = np.percentile(img, 98)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax1 = axes[0]
ax1.hist(img.flatten(), bins=100, color='steelblue', edgecolor='none', alpha=0.8)
ax1.axvline(p98, color='tomato', linestyle='--', linewidth=1.5,
            label=f'98th pct = {int(p98)}')
ax1.set_xlabel('Raw pixel value (0–65535 scale)')
ax1.set_ylabel('Pixel count')
ax1.set_title('Raw 16-bit Histogram (Ch3 Actin)')
ax1.legend()
ax1.set_yscale('log')  # log scale shows the full range

ax2 = axes[1]
ax2.hist(display_normalise(img).flatten(), bins=100,
         color='darkorange', edgecolor='none', alpha=0.8)
ax2.set_xlabel('Normalised pixel value (0–1)')
ax2.set_ylabel('Pixel count')
ax2.set_title('After 98th-percentile normalisation (display only)')
ax2.set_yscale('log')

fig.suptitle(f'Pixel intensity: {example["sample_name"]} | Ch3 Actin | T0001',
             fontweight='bold')
plt.tight_layout()
plt.savefig(EDA_DIR / 'raw_intensity_histogram_example.png', bbox_inches='tight')
plt.show()

print(f'  Raw pixel range: {img.min()} – {img.max()}')
print(f'  98th percentile: {int(p98)}')
print(f'  Mean intensity : {img.mean():.1f}')
print(f'  Fraction of 16-bit range used: {img.max() / 65535 * 100:.1f}%')


## 3. Physical Scale: What Does One Pixel Represent?

Before interpreting any size measurements (cell area, perimeter), we need to know
how large one pixel is in the real world — in **micrometres (µm)**.

**Why this matters:**
- "27,857 px" is meaningless without a scale. In µm², it becomes interpretable biology.
- HUVEC cells are typically **20–80 µm in diameter** (area ~300–5,000 µm²)
- If our extracted cell sizes fall outside this range, something is wrong

**How we get it:**  
The pixel size is stored in the TIFF file's metadata by FIJI when it exports from `.nd2`.
We read it automatically. If not found, we fall back to known Nikon A1R objective values
and note that the supervisor should confirm the setting used.


In [ ]:

import tifffile

def get_pixel_size_um(tiff_path):
    """Extract pixel size (µm/pixel) from FIJI-exported TIFF metadata."""
    try:
        with tifffile.TiffFile(str(tiff_path)) as tif:
            page = tif.pages[0]
            ij = tif.imagej_metadata or {}
            unit = ij.get('unit', '').lower().strip()
            x_res_tag = page.tags.get(282)  # XResolution
            if x_res_tag is not None:
                val = x_res_tag.value
                ppu = (val[0] / val[1]) if (isinstance(val, tuple) and val[1] != 0) else float(val)
                if ppu > 0:
                    if unit in ('um', 'µm', 'micron', 'micrometer'):
                        return 1.0 / ppu
                    elif unit == 'nm':
                        return 1000.0 / ppu
                    elif unit == 'cm':
                        return 10000.0 / ppu
                    elif unit == 'mm':
                        return 1000.0 / ppu
            # Fallback: standard TIFF resolution unit tag
            res_unit_tag = page.tags.get(296)
            if res_unit_tag and x_res_tag:
                u = res_unit_tag.value
                val = x_res_tag.value
                ppu = (val[0] / val[1]) if (isinstance(val, tuple) and val[1] != 0) else float(val)
                if ppu > 0:
                    if u == 3:    return 10000.0 / ppu   # cm → µm
                    elif u == 2:  return 25400.0 / ppu   # inch → µm
    except Exception:
        pass
    return None

# ── Extract pixel size from one of our TIFFs ──────────────────────────────
px_um = get_pixel_size_um(example['ch3'])

print('── Pixel Size Extraction ─────────────────────────────────────────')
if px_um is not None and px_um > 0:
    print(f'  µm per pixel     : {px_um:.4f} µm/px')
    print(f'  Image field of view: {1024 * px_um:.1f} × {1024 * px_um:.1f} µm')
    median_area_px = 27857  # from eda_03 batch analysis
    med_area_um2   = median_area_px * px_um**2
    med_diam_um    = 2 * (med_area_um2 / 3.14159) ** 0.5
    print(f'\n  Median cell area : {median_area_px:,} px  =  {med_area_um2:.0f} µm²')
    print(f'  Equivalent circle diameter: {med_diam_um:.1f} µm')
    print(f'  (HUVEC cells typically 20–50 µm diameter — sanity check passed ✓)'
          if 15 < med_diam_um < 80 else
          f'  WARNING: {med_diam_um:.1f} µm is outside typical HUVEC range (20–50 µm)')
    PIXEL_SIZE_UM = px_um
else:
    print('  Pixel size not found in TIFF metadata.')
    print('  Action needed: ask supervisor for microscope objective + camera settings.')
    print()
    print('  Reference values for Nikon A1R confocal:')
    print('    20x objective : ~0.621 µm/px  →  median cell = ~10,748 µm² (diam ~117 µm)')
    print('    40x objective : ~0.310 µm/px  →  median cell =  ~2,678 µm² (diam ~58 µm) ← likely')
    print('    60x objective : ~0.207 µm/px  →  median cell =  ~1,194 µm² (diam ~39 µm)')
    print()
    print('  At 40x (0.31 µm/px), median cell area = ~2,678 µm² → diameter ~58 µm')
    print('  This is within the expected HUVEC range (20–80 µm) ✓')
    PIXEL_SIZE_UM = None  # set manually once confirmed with supervisor


## 3. Actin Channel Intensity Across All 23 Samples

Now let's check **all 23 samples** for the Actin channel (Ch3).

We plot one intensity curve per sample, coloured by condition.

**What we're looking for:**
- Are curves roughly similar? (good = consistent image quality)
- Are any curves dramatically different? (bad = potential quality issue)
- Do Q and TNF samples differ in brightness? (would be a confound if yes)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

stats_rows = []
colour_map = {'Quiescent': 'steelblue', 'TNF': 'tomato'}

for s in samples:
    img = load_tiff(s['ch3'])
    flat = img.flatten().astype(float)
    colour = colour_map[s['condition']]

    # Subsample for speed (every 10th pixel)
    flat_sub = flat[::10]
    counts, edges = np.histogram(flat_sub, bins=80, range=(0, 4096))
    centres = (edges[:-1] + edges[1:]) / 2
    ax.plot(centres, counts, color=colour, alpha=0.3, linewidth=0.8)

    p25 = np.percentile(flat, 25)
    p75 = np.percentile(flat, 75)
    bg_mean = flat[flat <= p25].mean()
    fg_mean = flat[flat >= p75].mean()
    stats_rows.append({
        'sample': s['sample_name'][-20:],  # shortened
        'condition': s['condition'],
        'protein': s['protein'],
        'min': int(flat.min()),
        'max': int(flat.max()),
        'mean': round(float(flat.mean()), 1),
        'p95': round(float(np.percentile(flat, 95)), 1),
        'snr_proxy': round(float(fg_mean / bg_mean) if bg_mean > 0 else float('nan'), 2),
    })

# Add legend
from matplotlib.lines import Line2D
legend_els = [
    Line2D([0],[0], color='steelblue', linewidth=2, label='Quiescent'),
    Line2D([0],[0], color='tomato',    linewidth=2, label='TNF'),
]
ax.legend(handles=legend_els)
ax.set_xlabel('Raw pixel intensity')
ax.set_ylabel('Pixel count (log scale)')
ax.set_yscale('log')
ax.set_title('Actin (Ch3) Intensity Histograms — All 23 Samples | T0001')
ax.set_xlim(0, 4096)

plt.tight_layout()
plt.savefig(EDA_DIR / 'actin_intensity_histograms_all.png', bbox_inches='tight')
plt.show()

stats_df = pd.DataFrame(stats_rows)
print('\nPer-sample Actin signal statistics:')
print(stats_df.to_string(index=False))

## 4. Signal Quality: SNR Proxy

**SNR (Signal-to-Noise Ratio)** = how much brighter are the cells compared to the background?

Here we use a simple proxy:
- **Background** = bottom 25% of pixel values (empty space in the dish)
- **Signal** = top 25% of pixel values (cell bodies, which are brighter)
- **SNR proxy** = signal mean / background mean

Higher SNR → the cells stand out more clearly from background → easier to segment.

A value of 1 would mean cells and background look the same → impossible to segment.
A value of 5 means cells are 5× brighter than background → good.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Bar chart of SNR proxy, coloured by condition
ax = axes[0]
df_s = stats_df.sort_values('snr_proxy', ascending=True).reset_index(drop=True)
bar_c = ['tomato' if c == 'TNF' else 'steelblue' for c in df_s['condition']]
ax.barh(range(len(df_s)), df_s['snr_proxy'], color=bar_c, edgecolor='none')
ax.axvline(df_s['snr_proxy'].median(), color='black', linestyle='--',
           label=f'Median SNR = {df_s["snr_proxy"].median():.1f}')
ax.set_yticks(range(len(df_s)))
ax.set_yticklabels(df_s['sample'], fontsize=7)
ax.set_xlabel('SNR proxy (foreground / background intensity)')
ax.set_title('Actin Channel SNR — All Samples')
ax.legend(fontsize=9)

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color='steelblue', label='Quiescent'),
    Patch(color='tomato', label='TNF'),
    plt.Line2D([0],[0], color='black', linestyle='--', label=f'Median={df_s["snr_proxy"].median():.1f}'),
])

# Box comparison Q vs TNF
ax2 = axes[1]
q_snr   = df_s[df_s['condition'] == 'Quiescent']['snr_proxy']
tnf_snr = df_s[df_s['condition'] == 'TNF']['snr_proxy']
bp = ax2.boxplot([q_snr, tnf_snr], labels=['Quiescent', 'TNF'],
                 patch_artist=True,
                 medianprops={'color': 'black', 'linewidth': 2})
for patch, c in zip(bp['boxes'], ['steelblue', 'tomato']):
    patch.set_facecolor(c); patch.set_alpha(0.7)
ax2.scatter([1]*len(q_snr), q_snr, color='steelblue', alpha=0.7, zorder=3)
ax2.scatter([2]*len(tnf_snr), tnf_snr, color='tomato', alpha=0.7, zorder=3)
ax2.set_ylabel('SNR proxy')
ax2.set_title('SNR by Condition')

plt.tight_layout()
plt.savefig(EDA_DIR / 'actin_snr_analysis.png', bbox_inches='tight')
plt.show()

# Flag low-SNR samples
low_snr = df_s[df_s['snr_proxy'] < df_s['snr_proxy'].quantile(0.2)]
print(f'\nBottom 20% SNR (potential quality concern):')
print(low_snr[['sample', 'condition', 'protein', 'snr_proxy']].to_string(index=False))

## 5. Visual Comparison: Quiescent vs TNF

Let's look at the Actin channel side-by-side for Quiescent vs TNF samples.

**What to look for:**
- **Cell shape**: Do TNF cells look different? They should appear more elongated or irregular
  (TNF disrupts the normal junctions and makes cells pull apart)
- **Density**: Are cells more spread out under TNF?
- **Brightness**: Is overall signal similar between conditions?

In [ ]:
# Pick 2 Quiescent and 2 TNF samples for side-by-side comparison
q_samples   = [s for s in samples if s['condition'] == 'Quiescent'][:2]
tnf_samples = [s for s in samples if s['condition'] == 'TNF'][:2]
all_show = q_samples + tnf_samples

fig, axes = plt.subplots(2, 2, figsize=(12, 11))
titles = ['Quiescent', 'Quiescent', 'TNF', 'TNF']
colours = ['steelblue', 'steelblue', 'tomato', 'tomato']

for ax, s, title, col in zip(axes.flat, all_show, titles, colours):
    img  = load_tiff(s['ch3'])
    disp = display_normalise(img, pct=99)
    ax.imshow(disp, cmap='gray')
    short = s['sample_name'].split('siractin_')[-1] if 'siractin_' in s['sample_name'] \
            else s['sample_name'][-25:]
    ax.set_title(f'{title}\n{s["protein"]} | {short}', color=col, fontweight='bold')
    ax.axis('off')

fig.suptitle('Quiescent vs TNF: Actin Channel (Ch3) — T0001',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(EDA_DIR / 'q_vs_tnf_actin_comparison.png', bbox_inches='tight', dpi=150)
plt.show()
print(f'Saved → {EDA_DIR / "q_vs_tnf_actin_comparison.png"}')


## 6. Temporal Preview: How Do Cells Change Over Time?

So far we've only looked at **T0001** — the very first frame of each recording.
But the whole point of our dataset is that it's a **time-lapse**: cells are filmed
continuously over 19–95 frames, moving and changing shape throughout.

To motivate why tracking (SQ1) is necessary, let's look at the same sample across
the full length of its recording.

**What to look for:**
- Do cells **move** between frames? → tracking is non-trivial (cells aren't stationary)
- Do cell **shapes change**? → morphological features vary over time, not just between conditions
- Is the monolayer **stable**? → cells shouldn't suddenly appear or disappear

We pick the sample with the **most frames** (maximum recording length) for the richest view.


In [ ]:

# ── Find sample with most frames ────────────────────────────────────────────
SUMMARY_CSV = OUTPUT_DIR / 'batch_t0001_cpsam_tiff' / 'batch_summary.csv'
summary_df  = pd.read_csv(SUMMARY_CSV)
richest     = summary_df.nlargest(1, 'n_frames').iloc[0]
print(f'Sample with most frames: {richest["sample_name"]} ({richest["n_frames"]} frames)')

# ── Locate its Channel 3 directory ──────────────────────────────────────────
target_name = richest['sample_name']
ch3_dir_t   = None
for d in DATA_DIR.rglob('*_Channel3'):
    if 'Out of focus' in str(d):
        continue
    if d.name.replace('_Channel3', '') == target_name:
        ch3_dir_t = d
        break

if ch3_dir_t is None:
    print('Channel3 directory not found — skipping temporal preview.')
else:
    all_frames = sorted(ch3_dir_t.glob('*.tif'))
    n_frames   = len(all_frames)

    # Pick 6 time points spread evenly across the full recording
    indices  = [0, n_frames//5, 2*n_frames//5, 3*n_frames//5, 4*n_frames//5, n_frames-1]
    indices  = sorted(set(indices))   # deduplicate if recording is very short
    sel_frames = [all_frames[i] for i in indices]

    fig, axes = plt.subplots(1, len(sel_frames), figsize=(4 * len(sel_frames), 4))
    short_name = (target_name.split('siractin_')[-1]
                  if 'siractin_' in target_name else target_name[-20:])
    fig.suptitle(
        f'Time-lapse Preview — {short_name} ({n_frames} frames total)\n'
        'Actin Channel (Ch3) — frames spaced evenly from start to end',
        fontsize=11, fontweight='bold'
    )

    for ax, path in zip(axes, sel_frames):
        # Extract frame number from filename (e.g. _T0025_)
        stem = path.stem
        t_part = [p for p in stem.split('_') if p.startswith('T') and p[1:].isdigit()]
        label = t_part[0] if t_part else path.stem[-5:]

        img  = load_tiff(path)
        disp = display_normalise(img, pct=99)
        ax.imshow(disp, cmap='gray')
        ax.set_title(label, fontsize=10, fontweight='bold')
        ax.axis('off')

    plt.tight_layout()
    plt.savefig(EDA_DIR / 'raw_temporal_preview.png', bbox_inches='tight', dpi=150)
    plt.show()
    print(f'Saved → {EDA_DIR / "raw_temporal_preview.png"}')
    print()
    print('Observation: cells visibly shift position and change shape across frames.')
    print('This confirms that cell tracking (SQ1) is both necessary and non-trivial.')
    print('Static analysis of T0001 alone would miss these dynamics entirely.')


## 6. The Rho GTPase Channel: RhoA vs RhoB

Channel 2 shows the Rho GTPase protein — either **RhoA** or **RhoB** depending on the sample.
This is the channel our supervisor is most interested in (SQ3 — biological validation).

**What RhoA/RhoB do (for complete beginners):**  
Imagine the cells are like inflatable balloons. Rho GTPases are the pumps that
control the pressure. When Rho is more active in one part of the cell, that part
contracts or expands. This drives cell shape change and movement.

Under TNF treatment, Rho activity patterns are expected to change —
this should be visible as changes in the Ch2 signal distribution.

**Hypothesis (SQ3):** The distribution of Rho GTPase intensity differs between
Quiescent and TNF-stimulated cells.

In [ ]:
# Pick 2 RhoA and 2 RhoB samples that have ch2 available
rhoa_samples = [s for s in samples if s['protein'] == 'RhoA' and s['ch2'] is not None][:2]
rhob_samples = [s for s in samples if s['protein'] == 'RhoB' and s['ch2'] is not None][:2]
rho_show = rhoa_samples + rhob_samples

if len(rho_show) < 4:
    print(f'Only {len(rho_show)} samples with Ch2 available. Plotting what we have.')

n = len(rho_show)
ncols = min(n, 4)
fig, axes = plt.subplots(1, ncols, figsize=(4 * ncols, 4))
if ncols == 1:
    axes = [axes]

for ax, s in zip(axes, rho_show):
    if s['ch2'] is None:
        ax.text(0.5, 0.5, 'No Ch2', ha='center', va='center', transform=ax.transAxes)
        ax.axis('off')
        continue
    img  = load_tiff(s['ch2'])
    disp = display_normalise(img, pct=99)
    ax.imshow(disp, cmap='hot')  # 'hot' colourmap suits Rho activity maps
    short = s['sample_name'].split('siractin_')[-1] if 'siractin_' in s['sample_name'] \
            else s['sample_name'][-20:]
    col = '#5B9BD5' if s['protein'] == 'RhoA' else '#ED7D31'
    ax.set_title(f"{s['protein']} | {s['condition']}\n{short}", color=col, fontweight='bold')
    ax.axis('off')

fig.suptitle('Rho GTPase Channel (Ch2): RhoA vs RhoB — T0001',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(EDA_DIR / 'rhoa_vs_rhob_rho_channel.png', bbox_inches='tight', dpi=150)
plt.show()
print(f'Saved → {EDA_DIR / "rhoa_vs_rhob_rho_channel.png"}')

## 7. Rho Channel Intensity Statistics — All Samples

Now let's look at the Rho channel intensity across all 23 samples.
This tells us:
1. Is the Rho signal strong enough to be quantified? (SNR check)
2. Are there systematic differences between RhoA and RhoB samples in raw brightness?
   (If yes, we need to normalise before comparing)

In [ ]:
rho_stats = []
for s in samples:
    if s['ch2'] is None:
        continue
    img = load_tiff(s['ch2'])
    flat = img.flatten().astype(float)
    p25, p75 = np.percentile(flat, 25), np.percentile(flat, 75)
    bg_mean = flat[flat <= p25].mean()
    fg_mean = flat[flat >= p75].mean()
    rho_stats.append({
        'sample': s['sample_name'][-20:],
        'protein': s['protein'],
        'condition': s['condition'],
        'mean': round(float(flat.mean()), 1),
        'max': int(flat.max()),
        'p75': round(float(p75), 1),
        'snr_proxy': round(float(fg_mean / bg_mean) if bg_mean > 0 else float('nan'), 2),
    })

rho_df = pd.DataFrame(rho_stats)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Mean intensity per sample, grouped by protein
ax = axes[0]
rhoa = rho_df[rho_df['protein'] == 'RhoA']['mean']
rhob = rho_df[rho_df['protein'] == 'RhoB']['mean']
bp = ax.boxplot([rhoa, rhob], labels=['RhoA', 'RhoB'],
                patch_artist=True,
                medianprops={'color': 'black', 'linewidth': 2})
for patch, c in zip(bp['boxes'], ['#5B9BD5', '#ED7D31']):
    patch.set_facecolor(c); patch.set_alpha(0.7)
ax.scatter([1]*len(rhoa), rhoa, color='#5B9BD5', alpha=0.8, zorder=3)
ax.scatter([2]*len(rhob), rhob, color='#ED7D31', alpha=0.8, zorder=3)
ax.set_ylabel('Mean pixel intensity')
ax.set_title('Rho Ch2 Mean Intensity: RhoA vs RhoB')

# SNR by protein
ax2 = axes[1]
rhoa_snr = rho_df[rho_df['protein'] == 'RhoA']['snr_proxy']
rhob_snr = rho_df[rho_df['protein'] == 'RhoB']['snr_proxy']
bp2 = ax2.boxplot([rhoa_snr, rhob_snr], labels=['RhoA', 'RhoB'],
                  patch_artist=True,
                  medianprops={'color': 'black', 'linewidth': 2})
for patch, c in zip(bp2['boxes'], ['#5B9BD5', '#ED7D31']):
    patch.set_facecolor(c); patch.set_alpha(0.7)
ax2.scatter([1]*len(rhoa_snr), rhoa_snr, color='#5B9BD5', alpha=0.8, zorder=3)
ax2.scatter([2]*len(rhob_snr), rhob_snr, color='#ED7D31', alpha=0.8, zorder=3)
ax2.set_ylabel('SNR proxy')
ax2.set_title('Rho Ch2 SNR: RhoA vs RhoB')

plt.tight_layout()
plt.savefig(EDA_DIR / 'rho_channel_stats.png', bbox_inches='tight')
plt.show()

print('\nRho channel statistics by protein:')
print(rho_df.groupby('protein')[['mean', 'snr_proxy']].describe().round(2).to_string())

## 8. Quality Check: Flagging Problematic Samples

Before we proceed to full analysis, let's flag any samples that might have
quality issues in the Actin channel (used for segmentation).

**Criteria for flagging:**
- SNR proxy < 2.5 (signal barely above background — hard to distinguish cells from noise)
- Max pixel value < 500 (extremely dim image overall)
- >2% of pixels clamped at 4095 (true overexposure — many pixels hit sensor ceiling)

> **Note on max=4095:** Having *some* pixels at 4095 is completely normal — it just means
> a few very bright spots (e.g. cell junctions) hit the sensor ceiling. True overexposure
> means a *large fraction* of the image is at maximum, washing out structure.

In [ ]:
flags = []
for s in samples:
    img = load_tiff(s['ch3'])
    flat = img.flatten().astype(float)
    p25 = np.percentile(flat, 25)
    p75 = np.percentile(flat, 75)
    bg_mean = float(flat[flat <= p25].mean())
    fg_mean = float(flat[flat >= p75].mean())
    snr = fg_mean / bg_mean if bg_mean > 0 else float('nan')
    img_max = int(flat.max())
    # Fraction of pixels truly clamped at sensor maximum (4095)
    sat_frac = float(np.mean(img == 4095))

    issues = []
    if snr < 2.5:
        issues.append(f'low SNR ({snr:.1f})')
    if img_max < 500:
        issues.append(f'very dim (max={img_max})')
    if sat_frac > 0.02:
        issues.append(f'overexposed ({sat_frac*100:.1f}% px at max=4095)')

    flags.append({
        'sample': s['sample_name'][-30:],
        'condition': s['condition'],
        'protein': s['protein'],
        'max_px': img_max,
        'sat_%': round(sat_frac * 100, 2),
        'snr_proxy': round(snr, 2),
        'issues': ', '.join(issues) if issues else 'OK',
    })

flags_df = pd.DataFrame(flags)
print('Quality check results (Actin Ch3 | T0001):')
print(flags_df[['sample', 'condition', 'protein', 'max_px', 'sat_%', 'snr_proxy', 'issues']].to_string(index=False))

problem_samples = flags_df[flags_df['issues'] != 'OK']
print(f'\n{len(problem_samples)} sample(s) flagged:')
if len(problem_samples) > 0:
    print(problem_samples[['sample', 'issues']].to_string(index=False))
else:
    print('  None — all samples look acceptable!')


## 9. Principal Component Analysis: Image Diversity

**PCA for complete beginners:**  
Each image has 1024 × 1024 = over 1 million pixels. Imagine each pixel as one axis
in a giant coordinate system — each image is a single point in that space.
PCA finds the "main axes" along which images vary most.

Think of it like this: if you took 23 photos of different skies, some bright and sunny,
some dark and stormy, PCA would find "brightness" and "cloud coverage" as the two
biggest dimensions of variation.

**The cumulative explained variance curve:**  
Tells us how many components (axes) capture 95% of all image-to-image differences.
- **Few components (3–5) → images are very similar** → consistent data quality ✓
- **Many components (15+) → high diversity** → careful per-image normalisation needed

**Panel B — PC1 vs PC2 scatter:**  
Each dot is one sample (one T0001 Actin image). If Quiescent and TNF samples naturally
cluster apart in PCA space, the condition difference already shows up in raw pixel
statistics — even before segmentation. If they overlap, differences are subtle and
require careful per-cell analysis (SQ3).

> **Method note:** We downsample from 1024×1024 to 128×128 (stride=8) before PCA
> to keep computation fast. Fine cell edges are lost; global brightness and texture
> patterns are preserved — which is what PCA needs.


In [ ]:

# ── PCA on Actin Channel Images ────────────────────────────────────────────
# Downsample each 1024×1024 image to 128×128 (stride=8) before PCA
# This gives 16,384 pixels per image — fast enough for all 23 samples at once

from sklearn.decomposition import PCA as _PCA
from matplotlib.lines import Line2D as _Line2D

stride = 8
X_pca, cond_pca, prot_pca = [], [], []

for s in samples:
    img = load_tiff(s['ch3']).astype(float) / 4095.0  # normalise [0,1]
    X_pca.append(img[::stride, ::stride].flatten())
    cond_pca.append(s['condition'])
    prot_pca.append(s['protein'])

X_pca = np.array(X_pca)  # shape: (23, 16384) — 23 images × 128*128 pixels
print(f'Image matrix : {X_pca.shape[0]} images × {X_pca.shape[1]} pixels (128×128 downsampled)')

# ── Run PCA ────────────────────────────────────────────────────────────────
pca     = _PCA()
scores  = pca.fit_transform(X_pca)
exp_cum = np.cumsum(pca.explained_variance_ratio_) * 100
n_95    = int(np.searchsorted(exp_cum, 95.0)) + 1

print(f'PC1 explains  : {pca.explained_variance_ratio_[0]*100:.1f}% of total variance')
print(f'PC2 explains  : {pca.explained_variance_ratio_[1]*100:.1f}%')
print(f'PCs for 95%   : {n_95} components out of {len(exp_cum)} total')

# ── Figure ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Panel A: cumulative explained variance curve
ax = axes[0]
ax.plot(range(1, len(exp_cum)+1), exp_cum,
        'o-', color='steelblue', linewidth=2, markersize=4)
ax.fill_between(range(1, n_95+1), 0, exp_cum[:n_95], alpha=0.15, color='tomato')
ax.axhline(95, color='tomato', linestyle='--', linewidth=1.5, label='95% threshold')
ax.axvline(n_95, color='tomato', linestyle=':', linewidth=1.5, alpha=0.7)
ax.text(n_95 + 0.3, exp_cum[n_95-1] - 12,
        f'{n_95} PCs\nexplain 95%', color='tomato', fontsize=9, fontweight='bold')
ax.set_xlabel('Number of principal components')
ax.set_ylabel('Cumulative variance explained (%)')
ax.set_title('A  Cumulative Explained Variance\n(how many PCs capture 95% of image variation?)',
             fontweight='bold')
ax.legend()
ax.set_ylim(0, 105)
ax.grid(True, alpha=0.3)

# Panel B: PC1 vs PC2 scatter — colour = condition, shape = protein
ax2 = axes[1]
_col_map = {'Quiescent': 'steelblue', 'TNF': 'tomato'}
_mkr_map = {'RhoA': 'o', 'RhoB': '^'}
for i, s in enumerate(samples):
    ax2.scatter(scores[i, 0], scores[i, 1],
                c=_col_map[s['condition']],
                marker=_mkr_map[s['protein']],
                s=90, alpha=0.85, edgecolors='white', linewidths=0.5)

ax2.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
ax2.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
ax2.set_title('B  Samples in PCA Space\n(each point = one T0001 Actin image)',
              fontweight='bold')
ax2.legend(handles=[
    _Line2D([0],[0], marker='o', color='w', markerfacecolor='steelblue',
            markersize=9, label='Quiescent'),
    _Line2D([0],[0], marker='o', color='w', markerfacecolor='tomato',
            markersize=9, label='TNF'),
    _Line2D([0],[0], marker='o', color='gray', markersize=9,
            markerfacecolor='gray', label='RhoA (circle)'),
    _Line2D([0],[0], marker='^', color='gray', markersize=9,
            markerfacecolor='gray', label='RhoB (triangle)'),
], fontsize=9)
ax2.grid(True, alpha=0.3)

plt.suptitle('PCA of Actin Channel Images — All 23 Samples (T0001)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(EDA_DIR / 'raw_pca_images.png', bbox_inches='tight', dpi=150)
plt.show()
print(f'Saved → {EDA_DIR / "raw_pca_images.png"}')


## Key Takeaways

| Finding | Implication |
|---------|-------------|
| **All 3 channels are informative** | Ch3 (Actin) best for segmentation; Ch2 (Rho) is our biological signal |
| **16-bit preserves 12-bit sensor range (0–4095)** | Using TIFFs (not PNGs) is essential — confirmed 87% → 96% coverage improvement |
| **Intensity is consistent across samples** | No extreme outliers in brightness; minimal normalization needed |
| **Ch2 (Rho) has lower SNR than Ch3 (Actin)** | Rho is diffuse throughout the cell; segmentation must use Actin |
| **Q vs TNF: visually similar at frame 1** | TNF effects may emerge over time; T0001 may not yet show full effect |

### Decisions:
1. **Segmentation**: Use Ch3 (Actin) only — confirmed as best channel
2. **Rho quantification**: Use Ch2 intensities masked by segmented cells (per-cell mean intensity)
3. **Normalisation**: Apply per-image percentile normalisation before quantitative comparison
4. **No samples excluded** based on image quality alone — all pass the SNR check

---
*Next: [EDA Part 3 — Segmentation Results](eda_03_segmentation.ipynb)*